# Final Model Comparison

This notebook consolidates the completed October model experiments and selects the strongest candidate for the final modeling pipeline.

The comparison includes the required constant baselines and the main learned approaches tested by the team:

- Linear Regression
- Ridge Regression
- Random Forest
- HistGradientBoosting
- XGBoost
- CatBoost

All approaches use the same frozen claim-level split (`OCT-SPLIT-001`). Model-specific preprocessing, feature selection, and tuning were allowed, while the underlying training and validation claims remained fixed.

The primary selection metric is mean absolute error (MAE) in original `loss` units. Cross-validation stability, high-loss performance, model complexity, and feature count are used as supporting evidence.

No models are trained in this notebook. It only consolidates previously saved experimental results.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)


repo_root = Path.cwd()

while not (repo_root / "data" / "allstate_claims_data.csv").exists():
    if repo_root.parent == repo_root:
        raise FileNotFoundError("Could not locate repository root.")
    repo_root = repo_root.parent


controlled_dir = (
    repo_root
    / "notebooks"
    / "final-deliverables"
    / "October"
    / "Gate 3"
    / "controlled_model_comparison"
    / "model_comparison_results"
)

jonathan_dir = (
    repo_root
    / "notebooks"
    / "jonathan-deng"
    / "october_modeling_outputs"
)

liam_dir = (
    repo_root
    / "notebooks"
    / "liam-stapley"
    / "model_experiment_outputs"
)

output_dir = (
    repo_root
    / "notebooks"
    / "final-deliverables"
    / "October"
    / "Gate 3"
    / "final_model_comparison_results"
)

output_dir.mkdir(parents=True, exist_ok=True)

print("Repository:", repo_root)

NumPy: 2.5.3
Pandas: 3.0.6
Repository: /Users/Jonathan/Documents/GitHub/Allstate-1A-predicting-auto-claims-severity


## 1. Load Existing Model Evidence

Each modeling workflow saved its results independently. This section loads those artifacts rather than rerunning any model training.

Junaid's controlled comparison provides the accepted baselines, Ridge, and the previous leading HistGradientBoosting model. Jonathan's experiments add Linear Regression, Random Forest, and additional HistGradientBoosting configurations.

Liam's experiments add XGBoost and CatBoost, including the final feature-selected and tuned CatBoost candidate.

In [2]:
junaid_comparison = pd.read_csv(
    controlled_dir / "model_comparison.csv"
)

jonathan_comparison = pd.read_csv(
    jonathan_dir / "model_comparison.csv"
)

liam_initial = pd.read_csv(
    liam_dir / "full_feature_model_comparison.csv"
)

liam_final = pd.read_csv(
    liam_dir / "final_catboost_summary.csv"
)

liam_tail = pd.read_csv(
    liam_dir / "final_catboost_tail_diagnostics.csv"
)

print("Loaded all model comparison artifacts.")

Loaded all model comparison artifacts.


## 2. Baseline Reference

Two constant models provide the required reference points.

The training-mean baseline is the required project baseline. The training-median comparator is also important because MAE is minimized by a median-type constant prediction.

All learned models should substantially outperform both before being considered competitive.

In [3]:
MEAN_BASELINE_MAE = 1962.918654312225
MEDIAN_BASELINE_MAE = 1805.0863526621356

print(f"Mean baseline validation MAE:   {MEAN_BASELINE_MAE:.2f}")
print(f"Median baseline validation MAE: {MEDIAN_BASELINE_MAE:.2f}")

Mean baseline validation MAE:   1962.92
Median baseline validation MAE: 1805.09


## 3. Consolidated Model Comparison

The final comparison uses the strongest representative configuration from each model family rather than every intermediate tuning attempt.

The main fields are:

- **CV MAE:** Mean MAE across the five frozen training folds.
- **CV SD:** Fold-to-fold variability.
- **Validation MAE:** Performance on the shared validation claims.
- **Improvement vs. baseline:** Relative reduction in validation MAE.
- **Validation minus CV:** A simple diagnostic for differences between cross-validation and held-out validation performance.
- **Feature count:** Number of predictors retained by the final model.

In [4]:
final_catboost_val_mae = float(
    liam_final.iloc[0]["validation_mae"]
)

final_catboost_cv_mae = float(
    liam_final.iloc[0]["cv_mae_mean"]
)

final_catboost_cv_std = float(
    liam_final.iloc[0]["cv_mae_std"]
)

xgb_row = liam_initial[
    liam_initial["model"] == "XGBRegressor"
].iloc[0]

print(f"Final CatBoost validation MAE: {final_catboost_val_mae:.2f}")
print(f"Final CatBoost CV MAE:         {final_catboost_cv_mae:.2f}")
print(f"Final CatBoost CV SD:          {final_catboost_cv_std:.2f}")

Final CatBoost validation MAE: 1136.34
Final CatBoost CV MAE:         1153.58
Final CatBoost CV SD:          7.19


In [5]:
model_comparison = pd.DataFrame(
    [
        {
            "model": "Mean Constant",
            "variant": "Required baseline",
            "feature_count": 0,
            "cv_mae": 1968.12,
            "cv_std": 5.76,
            "validation_mae": MEAN_BASELINE_MAE,
            "complexity": "Very low",
        },
        {
            "model": "Median Constant",
            "variant": "Comparator",
            "feature_count": 0,
            "cv_mae": 1810.48,
            "cv_std": 6.20,
            "validation_mae": MEDIAN_BASELINE_MAE,
            "complexity": "Very low",
        },
        {
            "model": "Linear Regression",
            "variant": "Jonathan",
            "feature_count": 130,
            "cv_mae": 1310.31,
            "cv_std": np.nan,
            "validation_mae": 1290.13,
            "complexity": "Low",
        },
        {
            "model": "Ridge",
            "variant": "Junaid selected",
            "feature_count": 130,
            "cv_mae": 1308.55,
            "cv_std": 7.82,
            "validation_mae": 1290.43,
            "complexity": "Low",
        },
        {
            "model": "Random Forest",
            "variant": "Jonathan",
            "feature_count": 130,
            "cv_mae": 1243.54,
            "cv_std": np.nan,
            "validation_mae": 1223.51,
            "complexity": "Medium",
        },
        {
            "model": "HistGradientBoosting",
            "variant": "Jonathan lr=0.08",
            "feature_count": 130,
            "cv_mae": 1203.84,
            "cv_std": np.nan,
            "validation_mae": 1187.53,
            "complexity": "Medium",
        },
        {
            "model": "XGBRegressor",
            "variant": "Liam full 130",
            "feature_count": 130,
            "cv_mae": float(xgb_row["cv_mae"]),
            "cv_std": float(xgb_row["cv_std"]),
            "validation_mae": float(xgb_row["validation_mae"]),
            "complexity": "Medium/High",
        },
        {
            "model": "HistGradientBoosting",
            "variant": "Junaid selected",
            "feature_count": 130,
            "cv_mae": 1172.29,
            "cv_std": 6.29,
            "validation_mae": 1151.17,
            "complexity": "Medium",
        },
        {
            "model": "CatBoostRegressor",
            "variant": "Top 75, MAE, 500 iterations",
            "feature_count": 75,
            "cv_mae": final_catboost_cv_mae,
            "cv_std": final_catboost_cv_std,
            "validation_mae": final_catboost_val_mae,
            "complexity": "Medium/High",
        },
    ]
)

model_comparison["vs_mean_pct"] = (
    (MEAN_BASELINE_MAE - model_comparison["validation_mae"])
    / MEAN_BASELINE_MAE
    * 100
)

model_comparison["vs_median_pct"] = (
    (MEDIAN_BASELINE_MAE - model_comparison["validation_mae"])
    / MEDIAN_BASELINE_MAE
    * 100
)

model_comparison["validation_minus_cv"] = (
    model_comparison["validation_mae"]
    - model_comparison["cv_mae"]
)

model_comparison = (
    model_comparison
    .sort_values("validation_mae")
    .reset_index(drop=True)
)

model_comparison.round(2)

,model,variant,feature_count,cv_mae,cv_std,validation_mae,complexity,vs_mean_pct,vs_median_pct,validation_minus_cv
0,CatBoostRegressor,"Top 75, MAE, 500 iterations",75,1153.58,7.19,1136.34,Medium/High,42.11,37.05,-17.24
1,HistGradientBoosting,Junaid selected,130,1172.29,6.29,1151.17,Medium,41.35,36.23,-21.12
2,XGBRegressor,Liam full 130,130,1183.31,6.86,1162.00,Medium/High,40.80,35.63,-21.31
3,HistGradientBoosting,Jonathan lr=0.08,130,1203.84,NaN,1187.53,Medium,39.50,34.21,-16.31
4,Random Forest,Jonathan,130,1243.54,NaN,1223.51,Medium,37.67,32.22,-20.03
5,Linear Regression,Jonathan,130,1310.31,NaN,1290.13,Low,34.27,28.53,-20.18
6,Ridge,Junaid selected,130,1308.55,7.82,1290.43,Low,34.26,28.51,-18.12
7,Median Constant,Comparator,0,1810.48,6.20,1805.09,Very low,8.04,0.00,-5.39
8,Mean Constant,Required baseline,0,1968.12,5.76,1962.92,Very low,0.00,-8.74,-5.20


## 4. Overall Performance

Validation MAE remains the primary model-selection metric.

Cross-validation performance is considered alongside validation MAE to make sure the selected model is supported by repeated training-fold evidence rather than a single favorable validation result.

Lower MAE is better. Lower CV standard deviation indicates more consistent performance across training folds.

In [6]:
display_columns = [
    "model",
    "variant",
    "feature_count",
    "cv_mae",
    "cv_std",
    "validation_mae",
    "vs_mean_pct",
    "vs_median_pct",
    "validation_minus_cv",
    "complexity",
]

model_comparison[display_columns].round(
    {
        "cv_mae": 2,
        "cv_std": 2,
        "validation_mae": 2,
        "vs_mean_pct": 2,
        "vs_median_pct": 2,
        "validation_minus_cv": 2,
    }
)

,model,variant,feature_count,cv_mae,cv_std,validation_mae,vs_mean_pct,vs_median_pct,validation_minus_cv,complexity
0,CatBoostRegressor,"Top 75, MAE, 500 iterations",75,1153.58,7.19,1136.34,42.11,37.05,-17.24,Medium/High
1,HistGradientBoosting,Junaid selected,130,1172.29,6.29,1151.17,41.35,36.23,-21.12,Medium
2,XGBRegressor,Liam full 130,130,1183.31,6.86,1162.00,40.80,35.63,-21.31,Medium/High
3,HistGradientBoosting,Jonathan lr=0.08,130,1203.84,NaN,1187.53,39.50,34.21,-16.31,Medium
4,Random Forest,Jonathan,130,1243.54,NaN,1223.51,37.67,32.22,-20.03,Medium
5,Linear Regression,Jonathan,130,1310.31,NaN,1290.13,34.27,28.53,-20.18,Low
6,Ridge,Junaid selected,130,1308.55,7.82,1290.43,34.26,28.51,-18.12,Low
7,Median Constant,Comparator,0,1810.48,6.20,1805.09,8.04,0.00,-5.39,Very low
8,Mean Constant,Required baseline,0,1968.12,5.76,1962.92,0.00,-8.74,-5.20,Very low


## 5. CatBoost vs. Previous Best Model

Before the CatBoost experiments, the strongest controlled model was Junaid's HistGradientBoosting configuration.

Because the final recommendation should be based on meaningful improvement rather than only baseline comparisons, this section compares the final CatBoost model directly with that previous leader.

In [7]:
catboost_row = model_comparison[
    model_comparison["model"] == "CatBoostRegressor"
].iloc[0]

hgb_row = model_comparison[
    (model_comparison["model"] == "HistGradientBoosting")
    & (model_comparison["variant"] == "Junaid selected")
].iloc[0]

validation_advantage = (
    hgb_row["validation_mae"]
    - catboost_row["validation_mae"]
)

cv_advantage = (
    hgb_row["cv_mae"]
    - catboost_row["cv_mae"]
)

feature_reduction = (
    int(hgb_row["feature_count"])
    - int(catboost_row["feature_count"])
)

print(
    f"Validation MAE advantage: {validation_advantage:.2f}"
)

print(
    f"CV MAE advantage: {cv_advantage:.2f}"
)

print(
    f"Feature reduction: {feature_reduction} features"
)

Validation MAE advantage: 14.83
CV MAE advantage: 18.71
Feature reduction: 55 features


## 6. High-Loss Tail Diagnostics

Overall MAE can hide differences in performance across claim severity.

The team therefore evaluates errors across three validation-loss ranges defined using the training distribution:

- Below the training 90th percentile
- Between the training 90th and 99th percentiles
- At or above the training 99th percentile

These values are diagnostics rather than replacement selection metrics. The highest-loss segment contains relatively few claims, so small differences should be interpreted cautiously.

In [8]:
hgb_tail = pd.DataFrame(
    {
        "segment": [
            "loss < p90",
            "p90 to p99",
            "loss >= p99",
        ],
        "support": [
            25422,
            2547,
            279,
        ],
        "HGB_MAE": [
            874.2,
            3006.9,
            9446.5,
        ],
    }
)

tail_comparison = hgb_tail.merge(
    liam_tail[
        ["segment", "support", "mae"]
    ].rename(
        columns={"mae": "CatBoost_MAE"}
    ),
    on=["segment", "support"],
    how="inner",
)

tail_comparison["CatBoost_minus_HGB"] = (
    tail_comparison["CatBoost_MAE"]
    - tail_comparison["HGB_MAE"]
)

tail_comparison.round(2)

,segment,support,HGB_MAE,CatBoost_MAE,CatBoost_minus_HGB
0,loss < p90,25422,874.2,868.05,-6.15
1,p90 to p99,2547,3006.9,2976.91,-29.99
2,loss >= p99,279,9446.5,8779.90,-666.60


A negative `CatBoost_minus_HGB` value means CatBoost produced lower MAE for that severity range. A positive value means HistGradientBoosting performed better.

Tail performance is used to identify tradeoffs in the final recommendation, especially because very large claim errors can be operationally important even when they represent a small fraction of the validation set.

## 7. Stability Check

A strong validation result is more convincing when it agrees with performance across the five frozen training folds.

This section compares CV MAE, CV variability, and the gap between CV and validation performance for the strongest model families.

In [9]:
stability_comparison = model_comparison[
    model_comparison["model"].isin(
        [
            "CatBoostRegressor",
            "XGBRegressor",
            "HistGradientBoosting",
            "Ridge",
        ]
    )
][
    [
        "model",
        "variant",
        "cv_mae",
        "cv_std",
        "validation_mae",
        "validation_minus_cv",
    ]
].copy()

stability_comparison.round(2)

,model,variant,cv_mae,cv_std,validation_mae,validation_minus_cv
0,CatBoostRegressor,"Top 75, MAE, 500 iterations",1153.58,7.19,1136.34,-17.24
1,HistGradientBoosting,Junaid selected,1172.29,6.29,1151.17,-21.12
2,XGBRegressor,Liam full 130,1183.31,6.86,1162.00,-21.31
3,HistGradientBoosting,Jonathan lr=0.08,1203.84,NaN,1187.53,-16.31
6,Ridge,Junaid selected,1308.55,7.82,1290.43,-18.12


## 8. Selection Framework

The final recommendation uses a hierarchy rather than combining unrelated metrics into an arbitrary weighted score.

The criteria are considered in this order:

1. Validation MAE in original `loss` units
2. Five-fold CV MAE and stability
3. High-loss tail diagnostics
4. Prediction sanity and failure behavior
5. Model simplicity and feature count
6. Runtime and implementation cost

This keeps the agreed primary metric central while still accounting for model reliability and practical tradeoffs.

In [10]:
best_model = model_comparison.iloc[0]

print("Recommended model:")
print(best_model["model"])
print(best_model["variant"])

print(
    f"\nValidation MAE: "
    f"{best_model['validation_mae']:.2f}"
)

print(
    f"CV MAE: "
    f"{best_model['cv_mae']:.2f}"
)

print(
    f"CV SD: "
    f"{best_model['cv_std']:.2f}"
)

print(
    f"Improvement vs mean baseline: "
    f"{best_model['vs_mean_pct']:.2f}%"
)

print(
    f"Improvement vs median baseline: "
    f"{best_model['vs_median_pct']:.2f}%"
)

Recommended model:
CatBoostRegressor
Top 75, MAE, 500 iterations

Validation MAE: 1136.34
CV MAE: 1153.58
CV SD: 7.19
Improvement vs mean baseline: 42.11%
Improvement vs median baseline: 37.05%


## 9. Final Recommendation

The final recommended model is the feature-selected CatBoostRegressor using the top 75 ranked predictors.

CatBoost produced the strongest overall validation MAE among the tested approaches and also improved on the previous HistGradientBoosting leader across five-fold training cross-validation. This makes the result less dependent on a single validation partition.

The selected model also reduces the input set from 130 to 75 predictors. More aggressive feature reduction to 50 or 30 predictors worsened cross-validation performance, suggesting that the 75-feature subset provides a better balance between predictive signal and model simplicity.

High-loss diagnostics are reported separately so that the overall MAE improvement can be evaluated alongside performance on more severe claims.

The protected test partition remains unused and should be reserved for final evaluation after the model-selection process is complete.

## 10. Save Final Comparison Artifacts

The tables below are exported so the comparison can be independently reviewed and reused by later notebooks without rerunning any model experiments.

In [11]:
model_comparison.to_csv(
    output_dir / "final_model_comparison.csv",
    index=False,
)

tail_comparison.to_csv(
    output_dir / "final_tail_comparison.csv",
    index=False,
)

stability_comparison.to_csv(
    output_dir / "final_stability_comparison.csv",
    index=False,
)

print("Saved final model comparison artifacts to:")
print(output_dir)

Saved final model comparison artifacts to:
/Users/Jonathan/Documents/GitHub/Allstate-1A-predicting-auto-claims-severity/notebooks/final-deliverables/October/Gate 3/final_model_comparison_results
